# 07. 수요 대비 공급 분석

일별 이용현황으로 차량운행 대비 접수/탑승/대기시간 부담을 확인한다.


In [ ]:
import json
import platform
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.colors import Normalize
from matplotlib.patches import Polygon
from matplotlib.collections import PatchCollection
from IPython.display import display

# 한글 폰트 설정
if platform.system() == "Darwin":
    plt.rcParams["font.family"] = "AppleGothic"
elif platform.system() == "Windows":
    plt.rcParams["font.family"] = "Malgun Gothic"
else:
    plt.rcParams["font.family"] = "NanumGothic"

plt.rcParams["axes.unicode_minus"] = False

# 프로젝트 루트 설정
project_root = Path.cwd()
if project_root.name == "notebooks":
    project_root = project_root.parent

# 서울 자치구 지도 데이터 로드
map_path = project_root / "data" / "raw" / "seoul_municipalities_geo_simple.json"

with open(map_path, encoding="utf-8") as map_file:
    seoul_map = json.load(map_file)

In [ ]:
PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "data").exists() and (PROJECT_ROOT.parent / "data").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_DIR = PROJECT_ROOT / "data"
PROCESSED_DIR = DATA_DIR / "processed"

daily_path = PROCESSED_DIR / "서울시설공단_장애인콜택시 일별이용현황_20251231.csv"

df_daily = pd.read_csv(daily_path)
df_daily["기준일"] = pd.to_datetime(df_daily["기준일"], errors="coerce")

df_daily = df_daily[
    (df_daily["기준일"] >= "2025-01-01")
    & (df_daily["기준일"] < "2026-01-01")
].copy()

print(f"분석 행 수: {len(df_daily):,}")
print(f'기준일 변환 실패: {df_daily["기준일"].isna().sum():,}건')
display(df_daily.head())

## 공급 부담 지표 계산


In [ ]:
daily_supply_demand_summary = df_daily.copy()
daily_supply_demand_summary['차량1대당접수건'] = daily_supply_demand_summary['접수건'] / daily_supply_demand_summary['차량운행']
daily_supply_demand_summary['차량1대당탑승건'] = daily_supply_demand_summary['탑승건'] / daily_supply_demand_summary['차량운행']
daily_supply_demand_summary['접수대비탑승률(%)'] = daily_supply_demand_summary['탑승건'] / daily_supply_demand_summary['접수건'] * 100
daily_supply_demand_summary['미탑승건'] = daily_supply_demand_summary['접수건'] - daily_supply_demand_summary['탑승건']
daily_supply_demand_summary['요일'] = daily_supply_demand_summary['기준일'].dt.dayofweek.map(dict(enumerate(['월요일','화요일','수요일','목요일','금요일','토요일','일요일'])))

display(daily_supply_demand_summary.head())
display(daily_supply_demand_summary.describe())


## 일자별 차량운행/접수/탑승 추세


In [ ]:
fig, ax1 = plt.subplots(figsize=(16, 6))

ax1.plot(daily_supply_demand_summary['기준일'], daily_supply_demand_summary['접수건'], label='접수건', color='#4c78a8', alpha=0.8)
ax1.plot(daily_supply_demand_summary['기준일'], daily_supply_demand_summary['탑승건'], label='탑승건', color='#54a24b', alpha=0.8)
ax1.set_xlabel('기준일')
ax1.set_ylabel('건수')
ax1.grid(axis='y', alpha=0.3)

ax2 = ax1.twinx()
ax2.plot(daily_supply_demand_summary['기준일'], daily_supply_demand_summary['차량운행'], label='차량운행', color='#f58518', alpha=0.8)
ax2.set_ylabel('차량운행')

lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc='upper left')
plt.title('일자별 차량운행/접수건/탑승건 추세')
plt.tight_layout()
plt.show()


## 차량 1대당 접수건과 평균대기시간 관계


In [ ]:
plt.figure(figsize=(9, 6))
sns.scatterplot(
    data=daily_supply_demand_summary,
    x='차량1대당접수건',
    y='평균대기시간',
    hue='요일',
    alpha=0.75
)
plt.title('차량 1대당 접수건과 평균대기시간')
plt.xlabel('차량 1대당 접수건')
plt.ylabel('평균대기시간(분)')
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

correlation_summary = daily_supply_demand_summary[
    ['차량운행', '접수건', '탑승건', '차량1대당접수건', '차량1대당탑승건', '접수대비탑승률(%)', '평균대기시간']
].corr(numeric_only=True)

display(correlation_summary)


## 공급 부담이 큰 날짜 후보


In [ ]:
high_burden_days = (
    daily_supply_demand_summary
    .sort_values(['차량1대당접수건', '평균대기시간'], ascending=[False, False])
    .head(20)
)

long_wait_days = (
    daily_supply_demand_summary
    .sort_values('평균대기시간', ascending=False)
    .head(20)
)

low_conversion_days = (
    daily_supply_demand_summary
    .sort_values('접수대비탑승률(%)')
    .head(20)
)

print('차량 1대당 접수건 상위 날짜')
display(high_burden_days)

print('평균대기시간 상위 날짜')
display(long_wait_days)

print('접수 대비 탑승률 하위 날짜')
display(low_conversion_days)
